In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path.cwd()

chunks = []

for chunk in pd.read_csv(
    data_dir / "la.data.64.County.txt",
    sep="\t",
    dtype=str,
    chunksize=100_000
):
    chunk.columns = chunk.columns.str.strip()

    chunk["year"] = chunk["year"].str.strip()
    chunk["period"] = chunk["period"].str.strip()

    filtered = chunk[
        chunk["year"].isin(
            ["2019", "2020", "2021", "2022", "2023", "2024"]
        )
        & (chunk["period"] == "M13")
    ].copy()

    chunks.append(filtered)

bls = pd.concat(chunks, ignore_index=True)

for col in bls.select_dtypes(include="object").columns:
    bls[col] = bls[col].str.strip()

print("Total rows:", len(bls))
print("\nColumns:", bls.columns.tolist())
print("\nFirst five rows:")
print(bls.head())
print("\nRecords by year:")
print(bls["year"].value_counts().sort_index())

In [ ]:
bls["county_fips"] = bls["series_id"].str[5:10]

bls["measure_code"] = bls["series_id"].str[-2:]

print(
    bls[
        ["series_id", "county_fips", "measure_code", "year", "value"]
    ].head(10)
)

print("\nMeasurement counts:")
print(bls["measure_code"].value_counts())


In [ ]:

measure_names = {
    "03": "unemployment_rate",
    "04": "unemployment",
    "05": "employment",
    "06": "labor_force"
}

bls["measure"] = bls["measure_code"].map(measure_names)

bls["value"] = pd.to_numeric(bls["value"], errors="coerce")

bls_clean = bls.pivot(
    index=["county_fips", "year"],
    columns="measure",
    values="value"
).reset_index()

bls_clean.columns.name = None

bls_clean = bls_clean[
    [
        "county_fips",
        "year",
        "labor_force",
        "employment",
        "unemployment",
        "unemployment_rate"
    ]
]

print("Total county-year records:", len(bls_clean))
print(bls_clean.head())

In [ ]:
print("Missing values:")
print(bls_clean.isna().sum())

print("\nDuplicate county-year records:")
print(bls_clean.duplicated(["county_fips", "year"]).sum())

print("\nRecords by year:")
print(bls_clean["year"].value_counts().sort_index())

In [ ]:
missing_bls = bls_clean[
    bls_clean["unemployment_rate"].isna()
]

print("Records with missing unemployment rates:", len(missing_bls))

print("\nMissing records by year:")
print(missing_bls["year"].value_counts().sort_index())

print("\nSample missing records:")
print(missing_bls.head(15))

measure_cols = [
    "labor_force",
    "employment",
    "unemployment",
    "unemployment_rate"
]

print("\nRecords missing all four measurements:")
print(bls_clean[measure_cols].isna().all(axis=1).sum())

In [ ]:
valid_bls = bls_clean.dropna(
    subset=[
        "labor_force",
        "employment",
        "unemployment",
        "unemployment_rate"
    ]
).copy()

valid_bls["labor_force_difference"] = (
    valid_bls["labor_force"]
    - valid_bls["employment"]
    - valid_bls["unemployment"]
)

print("Labor force differences:")
print(valid_bls["labor_force_difference"].describe())

valid_bls["calculated_rate"] = (
    valid_bls["unemployment"]
    / valid_bls["labor_force"]
    * 100
)

valid_bls["rate_difference"] = (
    valid_bls["unemployment_rate"]
    - valid_bls["calculated_rate"]
).abs()

print("\nUnemployment rate differences:")
print(valid_bls["rate_difference"].describe())

In [ ]:

bls_final = bls_clean[
    ~bls_clean["county_fips"].str.startswith("72")
].copy()

print("Original records:", len(bls_clean))
print("Records after excluding Puerto Rico:", len(bls_final))

print("\nMissing values:")
print(bls_final.isna().sum())

print("\nDuplicate county-year records:")
print(bls_final.duplicated(["county_fips", "year"]).sum())

print("\nRecords by year:")
print(bls_final["year"].value_counts().sort_index())

output_dir = Path.cwd() / "processed"
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "bls_county_unemployment_2019_2024.csv"

bls_final.to_csv(output_file, index=False)

print("\nSaved to:", output_file)